In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import datetime
import sys
# %matplotlib notebook
%matplotlib inline

In [ ]:
matplotlib.__version__

In [ ]:
def get_seconds(duration_h_m_s):
    try:
        t_parse = datetime.datetime.strptime(duration_h_m_s,"%H:%M:%S")
    except:
        sys.stderr.write("Error parsing: '{}'".format(duration_h_m_s))
        raise
    t_delta = datetime.timedelta(hours=t_parse.hour, minutes=t_parse.minute, seconds=t_parse.second)
    t_sec = t_delta.total_seconds()
    return t_sec

In [ ]:
df_tsv = pd.read_csv("Fellowship of the Ring - page-to-timestamp.tsv",sep='\t')

In [ ]:
df_tsv

In [ ]:
def get_duration_s(maybe_str):
    if maybe_str is not np.nan:
        t_s = get_seconds(maybe_str)
    else:
        t_s = np.nan
    return t_s
page_number_raw = df_tsv['Cumulative page number']
t_start_str = df_tsv['Movie start']
t_stop_str = df_tsv['Movie stop']
page_number_l = []
t_start_l = []
t_stop_l = []
for page, t1, t2 in zip(page_number_raw, t_start_str, t_stop_str):
    page_number_l.append(page)
    t_start_l.append(get_duration_s(t1))
    t_stop_l.append(get_duration_s(t2))
page_number = np.array(page_number_l)
t_start = np.array(t_start_l)
t_stop = np.array(t_stop_l)
t_mid = (t_start + t_stop)/2
df_plot = pd.DataFrame(data={
    'page_number':page_number,
    't_start':t_start,
    't_stop':t_stop,
    't_mid':t_mid,
})

In [ ]:
page_number = df_tsv['Cumulative page number']
t_start = pd.to_datetime(df_tsv['Movie start'], format='%H:%M:%S').dt.time
t_stop = pd.to_datetime(df_tsv['Movie stop'], format='%H:%M:%S').dt.time
# t_interval = pd.Interval(left=t_start, right=t_stop, closed='both')
# t_mid = t_interval.mid
# df_plot = pd.DataFrame(data={
#     'page_number':page_number,
#     't_start':t_start,
#     't_stop':t_stop,
#     't_mid':t_mid,
# })

In [ ]:
fig, ax = plt.subplots(constrained_layout=True)
ax.plot(df_plot.page_number, df_plot.t_start, '.');
# yfmt = matplotlib.dates.DateFormatter('%H:%M:%S')
# ax.yaxis.set_major_formatter(yfmt)
# fig.autofmt_ydate();
ax.set_xlabel("Page number of book")
ax.set_ylabel("Duration into movie");
ax.set_title("The Fellowship of the Ring");

TODO:

- [ ] Add vertical lines for chapter breaks


## Emma messing around

In [ ]:
# !pip install plotly
# !pip install pyjanitor
from plotly import express, offline
from plotly import graph_objects as go
from janitor import clean_names
import time 

In [ ]:
offline.init_notebook_mode()

In [ ]:
lotr = clean_names(pd.read_csv("Fellowship of the Ring - page-to-timestamp.tsv",sep='\t'))

In [ ]:
lotr

In [ ]:
lotr.info()

In [ ]:
def clean_hhmmss(tm):
    if tm is None:
        return pd.NaT
    elif type(tm) == float:
        return pd.NaT 
    else:
        return pd.to_datetime(tm, errors = 'coerce')

In [ ]:
def get_midpoint(start, stop):
    if (start is None or stop is None or type(start) == float or type(stop) == float):
        return pd.NaT
    else:
        clean_start = clean_hhmmss(min(start,stop))
        clean_stop = clean_hhmmss(max(stop, stop))
        duration = clean_stop - clean_start
        midpoint = clean_start + (duration/2)
        return midpoint.strftime('%H:%M:%S')
    
    

In [ ]:
clean_hhmmss(lotr['movie_start'][2])

In [ ]:
# lotr.assign(
#     clean_movie_start = np.where(pd.isna(lotr['movie_start']), pd.NaT, pd.to_datetime(lotr['movie_start'], '%H:%M:%S'))
# )

In [ ]:
lotr['clean_movie_start'] = lotr.apply(lambda row: clean_hhmmss(row['movie_start']), axis = 1)
lotr['clean_movie_stop'] = lotr.apply(lambda row: clean_hhmmss(row['movie_stop']), axis = 1)

In [ ]:
lotr = lotr.assign(
    duration = lotr['clean_movie_stop'] - lotr['clean_movie_start']
)

In [ ]:
lotr['midpoint'] = lotr.apply(lambda row: get_midpoint(row['movie_start'],row['movie_stop']), axis = 1)
lotr['midpoint_str'] = lotr['midpoint'].to_string()

In [ ]:
lotr

In [ ]:
lotr.info()

In [ ]:
express.scatter(
    lotr,
    x = 'book_page_number',
    y = 'movie_start'
)

In [ ]:
express.scatter(
    lotr,
    x = 'book_page_number',
    y = 'midpoint'
).update_yaxes(tickformat="%H:%M:%S")

In [ ]:
express.scatter(
    lotr,
    x = 'book_page_number',
    y = 'midpoint_str'
)